# conditional-hparam-branch — worked example 2: Add the weight-decay term to the gradient only when it is nonzero

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conditional-hparam-branch`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Decoupled weight decay folds a `weight_decay * param` term into the gradient before the optimizer step. When `weight_decay == 0` the gradient must pass through untouched — otherwise you'd allocate and add a zero tensor every step and risk tiny floating-point drift. The standard pattern guards the add with `if weight_decay != 0`.

## Worked solution

1. **Function shape.** `apply_weight_decay(grad, param, weight_decay)` returns the gradient that the step will actually use.
2. **Guard the only operation that changes the gradient.** `if weight_decay != 0:` we compute `grad = grad + weight_decay * param`. The new gradient is the original plus an L2-penalty pull toward zero.
3. **The zero path is a pure pass-through.** No `else` body is needed — we simply `return grad`. Because we never created `weight_decay * param`, the returned tensor IS the input gradient, so a test comparing against vanilla SGD sees bit-identical numbers.
4. **Why branch instead of always adding?** Multiplying by zero and adding is mathematically a no-op but not a *computational* no-op: it costs a tensor allocation and can introduce `-0.0`/rounding artifacts in fused kernels. The conditional makes `weight_decay=0` a true skip.

In [ ]:
def apply_weight_decay(grad, param, weight_decay):
    if weight_decay != 0:
        grad = grad + weight_decay * param
    return grad

t.manual_seed(0)
param = t.tensor([2.0, -4.0, 6.0])
grad = t.tensor([1.0, 1.0, 1.0])

g_off = apply_weight_decay(grad, param, 0.0)
g_on = apply_weight_decay(grad, param, 0.01)
print('weight_decay=0   grad:', g_off.tolist())
print('weight_decay=0.01 grad:', g_on.tolist())